In [1]:
import geopandas as gpd
import os
import requests
from shapely.validation import make_valid

# Cargar el límite oficial de Usaquén
usaquen_oficial = gpd.read_file("../data/raw/limites/loca.json")

usaquen_filtrado = usaquen_oficial[
    usaquen_oficial["LocNombre"] == "USAQUEN"
]

usaquen_filtrado = usaquen_filtrado.to_crs(epsg=4326)

usaquen_polygon = usaquen_filtrado.geometry.iloc[0]

if not usaquen_polygon.is_valid:
    usaquen_polygon = make_valid(usaquen_polygon)

print(f"Polígono válido: {usaquen_polygon.is_valid}")
print(usaquen_polygon.geom_type)

Polígono válido: True
Polygon


In [2]:
coords = list(usaquen_polygon.exterior.coords)

poly_coords = " ".join(
    f"{lat} {lon}"
    for lon, lat in coords
)

print(poly_coords[:500])

4.6645853120000424 -74.01116193799993 4.664600325000038 -74.01116635999995 4.664670392000062 -74.0111970449999 4.6647556270000905 -74.0112404919999 4.664802361000056 -74.01126767399995 4.664843971000039 -74.01128191799995 4.664872431000049 -74.01129356699994 4.664905297000075 -74.01130481899992 4.664934909000067 -74.01131778699994 4.664984942000046 -74.01134688799993 4.665016592000086 -74.0113534969999 4.665055760000087 -74.01137346099995 4.665099846000089 -74.01138855699992 4.665135217000056 -7


In [ ]:
ruta_archivo = '../data/processed/usaquen.osm.xml'

query = f"""
[out:xml][timeout:180];
(
  way["highway"](poly:"{poly_coords}");
);
(._;>;);
out meta;
"""

response = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={'data': query.encode('utf-8')},
    headers={"User-Agent": "MiAplicacionGeografica/1.0"},
    timeout=300
)

if response.status_code == 200:
    #Guarda el contenido en disco
    with open(ruta_archivo, 'wb') as f:
        f.write(response.content)
    
    print(f"Archivo guardado en: {os.path.abspath(ruta_archivo)}\n")

    #primeras líneas
    print("Contenido del archivo")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        for i in range(10):
            print(f.readline(), end='')
else:
    print(f"Error HTTP {response.status_code}: {response.text[:200]}")

Archivo guardado en: c:\Users\USUARIO\Downloads\simulacion-movilidad-urbana\data\processed\usaquen.osm.xml

Contenido del archivo
<?xml version="1.0" encoding="UTF-8"?>
<osm version="0.6" generator="Overpass API 0.7.62.11 87bfad18">
<note>The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.</note>
<meta osm_base="2026-08-21T18:47:06Z"/>

  <node id="253845462" lat="4.6752483" lon="-74.0243597" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845500" lat="4.6752084" lon="-74.0244588" version="12" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253845875" lat="4.6757979" lon="-74.0264972" version="18" timestamp="2026-01-01T21:02:09Z" changeset="176719144" uid="1759764" user="Facalderonm"/>
  <node id="253846345" lat="4.6680374" lon="-74.0124907" version="9" timestamp="2020-06-01T15:55:02Z" changeset="86053783" uid="

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

OSM_FILE = "../data/processed/usaquen.osm.xml"
NET_FILE = "../data/processed/usaquen.net.xml"

In [3]:
tree_osm = ET.parse(OSM_FILE)
root_osm = tree_osm.getroot()

ways = []

for way in root_osm.findall("way"):
    tags = {
        tag.attrib["k"]: tag.attrib["v"]
        for tag in way.findall("tag")
    }

    if "highway" in tags:
        ways.append({
            "id": way.attrib["id"],
            **tags
        })

osm = pd.DataFrame(ways)

print(f"Total de vías OSM: {len(osm):,}")

print("\nTipos de highway:")
print(osm["highway"].value_counts())

Total de vías OSM: 12,310

Tipos de highway:
highway
footway           3160
residential       3039
service           2724
primary            972
secondary          657
tertiary           386
cycleway           257
steps              163
primary_link       144
pedestrian         142
trunk              141
proposed           124
path                86
corridor            81
construction        74
trunk_link          58
track               32
secondary_link      24
bus_stop            20
unclassified        16
tertiary_link        6
bridleway            2
services             1
platform             1
Name: count, dtype: int64


In [4]:
HIGHWAYS_VEHICULARES = [
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "unclassified",
    "residential",
    "living_street",
    "service"
]

osm_veh = osm[
    osm["highway"].isin(HIGHWAYS_VEHICULARES)
].copy()

print(f"Vías vehiculares: {len(osm_veh):,}")

Vías vehiculares: 8,167


In [5]:
total_veh = len(osm_veh)

sin_maxspeed = osm_veh["maxspeed"].isna().sum()
sin_lanes = osm_veh["lanes"].isna().sum()

print("Atributos faltantes en vías vehiculares:")

print(
    f"Sin maxspeed: {sin_maxspeed:,} "
    f"({sin_maxspeed / total_veh * 100:.2f}%)"
)

print(
    f"Sin lanes: {sin_lanes:,} "
    f"({sin_lanes / total_veh * 100:.2f}%)"
)

Atributos faltantes en vías vehiculares:
Sin maxspeed: 5,255 (64.34%)
Sin lanes: 4,034 (49.39%)


In [6]:
print("\nValores de maxspeed:")
print(osm_veh["maxspeed"].value_counts(dropna=False))

print("\nValores de lanes:")
print(osm_veh["lanes"].value_counts(dropna=False))


Valores de maxspeed:
maxspeed
NaN    5255
30     2498
50       94
40       94
10       93
20       69
60       64
Name: count, dtype: int64

Valores de lanes:
lanes
NaN    4034
2      3027
3       559
1       462
4        68
5        13
6         3
20        1
Name: count, dtype: int64


In [7]:
osm_veh["maxspeed_num"] = pd.to_numeric(
    osm_veh["maxspeed"],
    errors="coerce"
)

osm_veh["lanes_num"] = pd.to_numeric(
    osm_veh["lanes"],
    errors="coerce"
)

analisis_faltantes = (
    osm_veh
    .groupby("highway")
    .agg(
        tramos=("id", "count"),
        sin_maxspeed=("maxspeed_num", lambda x: x.isna().sum()),
        sin_lanes=("lanes_num", lambda x: x.isna().sum())
    )
)

analisis_faltantes["pct_sin_maxspeed"] = (
    analisis_faltantes["sin_maxspeed"] /
    analisis_faltantes["tramos"] * 100
)

analisis_faltantes["pct_sin_lanes"] = (
    analisis_faltantes["sin_lanes"] /
    analisis_faltantes["tramos"] * 100
)

analisis_faltantes.round(2)

,tramos,sin_maxspeed,sin_lanes,pct_sin_maxspeed,pct_sin_lanes
highway,,,,,
primary,972,574,0,59.05,0.00
primary_link,144,120,7,83.33,4.86
residential,3039,1431,1413,47.09,46.50
secondary,657,261,2,39.73,0.30
secondary_link,24,17,1,70.83,4.17
service,2724,2588,2588,95.01,95.01
tertiary,386,141,5,36.53,1.30
tertiary_link,6,5,2,83.33,33.33
trunk,141,71,0,50.35,0.00


In [8]:
tree_net = ET.parse(NET_FILE)
root_net = tree_net.getroot()

edges = []

for edge in root_net.findall("edge"):

    if edge.attrib.get("function") == "internal":
        continue

    lanes = edge.findall("lane")

    edges.append({
        "id": edge.attrib["id"],
        "type": edge.attrib.get("type"),
        "num_lanes": len(lanes),
        "speed": [
            float(lane.attrib["speed"])
            for lane in lanes
        ]
    })

net_edges = pd.DataFrame(edges)

print(f"Edges SUMO: {len(net_edges):,}")

Edges SUMO: 28,070


In [10]:
net_edges["speed_mean"] = net_edges["speed"].apply(np.mean)
net_edges["speed_kmh"] = net_edges["speed_mean"] * 3.6

print("Velocidades finales:")

print(
    net_edges["speed_kmh"].describe()
)

print("\nCarriles por edge:")

print(
    net_edges["num_lanes"].value_counts().sort_index()
)

Velocidades finales:
count    28070.000000
mean        28.376308
std         21.106055
min          5.004000
25%         10.008000
50%         20.016000
75%         29.988000
max        100.008000
Name: speed_kmh, dtype: float64

Carriles por edge:
num_lanes
1     25174
2      2022
3       754
4        89
5        19
6         9
7         1
10        2
Name: count, dtype: int64


In [11]:
resumen_tipos = (
    net_edges
    .groupby("type")
    .agg(
        edges=("id", "count"),
        velocidad_promedio_kmh=("speed_kmh", "mean"),
        carriles_promedio=("num_lanes", "mean")
    )
    .sort_values("edges", ascending=False)
)

resumen_tipos.round(2)

,edges,velocidad_promedio_kmh,carriles_promedio
type,,,
highway.residential,8727,39.12,1.06
highway.footway,7367,10.03,1.00
highway.service,5870,19.83,1.00
highway.primary,1326,66.00,2.42
highway.cycleway,1187,20.02,1.00
highway.secondary,985,56.10,1.79
highway.tertiary,968,45.67,1.17
highway.pedestrian,368,10.01,1.02
highway.path,276,20.02,1.00


In [12]:
def obtener_permisos(lane):
    return {
        "allow": lane.attrib.get("allow"),
        "disallow": lane.attrib.get("disallow")
    }


allow_disallow = []

for edge in root_net.findall("edge"):

    if edge.attrib.get("function") == "internal":
        continue

    for lane in edge.findall("lane"):
        allow_disallow.append({
            "edge_id": edge.attrib["id"],
            "type": edge.attrib.get("type"),
            "allow": lane.attrib.get("allow"),
            "disallow": lane.attrib.get("disallow")
        })

permisos = pd.DataFrame(allow_disallow)

print("Allow:")
print(permisos["allow"].value_counts(dropna=False).head(20))

print("\nDisallow:")
print(permisos["disallow"].value_counts(dropna=False).head(20))

Allow:
allow
NaN                                    16376
pedestrian                              7986
pedestrian delivery bicycle             5810
pedestrian bicycle                      1062
bicycle                                  427
pedestrian motorcycle moped bicycle      158
bus bicycle                              155
emergency authority bus bicycle           24
delivery bicycle                          14
Name: count, dtype: int64

Disallow:
disallow
NaN                                                                                                                                  15636
tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone                       14711
pedestrian tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone              828
pedestrian bicycle tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair sco

In [13]:
crossings = []

for node in root_osm.findall("node"):

    tags = {
        tag.attrib["k"]: tag.attrib["v"]
        for tag in node.findall("tag")
    }

    if tags.get("highway") == "crossing":
        crossings.append({
            "id": node.attrib["id"],
            **tags
        })

print(f"Cruces peatonales registrados en OSM: {len(crossings):,}")

Cruces peatonales registrados en OSM: 2,134


In [14]:
sidewalk_cols = [
    col for col in osm.columns
    if col.startswith("sidewalk")
]

print("Atributos de acera encontrados:")

for col in sidewalk_cols:
    print(f"\n{col}")
    print(osm[col].value_counts(dropna=False).head(10))

Atributos de acera encontrados:

sidewalk
sidewalk
NaN         11936
separate      314
both           38
right          14
no              8
Name: count, dtype: int64

sidewalk:right
sidewalk:right
NaN         12298
separate       10
no              2
Name: count, dtype: int64

sidewalk:left
sidewalk:left
NaN         12299
no              9
separate        2
Name: count, dtype: int64

sidewalk:both
sidewalk:both
NaN    12309
yes        1
Name: count, dtype: int64


In [16]:
print("VALIDACIÓN FINAL DE LA RED:")

print(f"Nodos cargados en NetEdit: 12,989")
print(f"Edges SUMO: {len(net_edges):,}")
print(f"Tipos de vía SUMO: {net_edges['type'].nunique()}")
print(f"Cruces OSM registrados: {len(crossings):,}")

print("\nTipos de vía:")
print(net_edges["type"].value_counts())

print("\nLa red fue revisada visualmente en NetEdit.")

VALIDACIÓN FINAL DE LA RED:
Nodos cargados en NetEdit: 12,989
Edges SUMO: 28,070
Tipos de vía SUMO: 19
Cruces OSM registrados: 2,134

Tipos de vía:
type
highway.residential       8727
highway.footway           7367
highway.service           5870
highway.primary           1326
highway.cycleway          1187
highway.secondary          985
highway.tertiary           968
highway.pedestrian         368
highway.path               276
highway.steps              237
highway.trunk              235
highway.primary_link       164
highway.track              162
highway.trunk_link          95
highway.unclassified        77
highway.secondary_link      18
highway.bridleway            4
highway.tertiary_link        3
highway.service|psv          1
Name: count, dtype: int64

La red fue revisada visualmente en NetEdit.


# F2.5 — Preparación de datos (ETL)

## Conversión de la red vial de OSM a SUMO

**Responsable:** Santiago Chitiva  
**Fase:** 2 - Preparación de datos (ETL)  
**Semana:** 3-4
**Prerrequisito:** F1.3 (cerrado)

### Objetivo

Convertir la red vial de OSM validada en F1.3 al formato nativo de SUMO mediante `netconvert`, ampliando el alcance para considerar vehículos, bicicletas y peatones.


## 1. Obtención de la red vial desde OSM

El límite oficial de la localidad de Usaquén se obtiene desde:

```text
../data/raw/limites/loca.json
````

Se filtra la localidad mediante el atributo `LocNombre == "USAQUEN"` y se transforma el sistema de coordenadas a EPSG:4326.

Luego se valida la geometría mediante `make_valid()` y se utiliza el polígono resultante para realizar una consulta a Overpass API.

La consulta utilizada fue:

```text
[out:xml][timeout:180];
(
  way["highway"](poly:"...");
);
(._;>;);
out meta;
```

Esta consulta obtiene las vías (`way`) etiquetadas con `highway` dentro del polígono de Usaquén y descarga también los nodos asociados mediante:

```text
(._;>;);
```

El resultado se almacena como:

```text
data/processed/usaquen.osm.xml
```

### Resultado

La red OSM obtenida contiene:

* **12.310 vías**
* Infraestructura vehicular, peatonal y ciclística.
* 23 categorías de `highway` registradas.

Principales tipos encontrados:

| Highway        | Cantidad |
| -------------- | -------: |
| footway        |    3.160 |
| residential    |    3.039 |
| service        |    2.724 |
| primary        |      972 |
| secondary      |      657 |
| tertiary       |      386 |
| cycleway       |      257 |
| steps          |      163 |
| primary_link   |      144 |
| pedestrian     |      142 |
| trunk          |      141 |
| proposed       |      124 |
| path           |       86 |
| corridor       |       81 |
| construction   |       74 |
| trunk_link     |       58 |
| track          |       32 |
| secondary_link |       24 |
| bus_stop       |       20 |
| unclassified   |       16 |
| tertiary_link  |        6 |
| bridleway      |        2 |
| services       |        1 |
| platform       |        1 |


## 2. Inclusión de modos no motorizados

Se revisó el `custom_filter` utilizado en F1.3, ya que estaba orientado solo a vehículos.

Debido al alcance de la simulación, se decidió conservar también infraestructura para:

* Vehículos motorizados.
* Bicicletas.
* Peatones.

Entre los tipos considerados se encuentran:

* `footway`
* `path`
* `cycleway`
* `pedestrian`
* `steps`
* `track`
* `corridor`
* `bridleway`

La red se obtuvo directamente desde Overpass API utilizando las vías etiquetadas con `highway`.


## 3. Conversión a red SUMO

La red OSM fue convertida mediante `netconvert`:

```bash
netconvert --osm-files data/processed/usaquen.osm.xml ^
  --output-file data/processed/usaquen.net.xml ^
  --geometry.remove ^
  --roundabouts.guess ^
  --ramps.guess ^
  --junctions.join ^
  --tls.guess-signals ^
  --tls.discard-simple
```

El archivo generado es:

```text
data/processed/usaquen.net.xml
```

La red fue abierta en NetEdit para realizar una revisión visual.


## 4. Análisis de atributos originales de OSM

Se analizaron los atributos `maxspeed` y `lanes` de las vías vehiculares originales.

```text
Total de vías OSM: 12.310
Vías vehiculares: 8.167
```

### `maxspeed`

```text
Sin maxspeed: 5.255
Porcentaje: 64,34 %
```

Valores encontrados:

| maxspeed | Cantidad |
| -------- | -------: |
| 30       |    2.498 |
| 50       |       94 |
| 40       |       94 |
| 10       |       93 |
| 20       |       69 |
| 60       |       64 |

### `lanes`

```text
Sin lanes: 4.034
Porcentaje: 49,39 %
```

Valores encontrados:

| lanes | Cantidad |
| ----- | -------: |
| 2     |    3.027 |
| 3     |      559 |
| 1     |      462 |
| 4     |       68 |
| 5     |       13 |
| 6     |        3 |
| 20    |        1 |

### Interpretación

La ausencia de `maxspeed` o `lanes` en OSM no significa que esos atributos estén ausentes en la red final de SUMO.

`netconvert` asigna valores y características por defecto a los edges generados. Esto se comprobó directamente sobre el archivo `.net.xml`.

Además, las vías exclusivamente peatonales o ciclísticas no necesariamente requieren una velocidad vehicular original de OSM, ya que no están destinadas al tráfico motorizado.

Por esta razón, el análisis de atributos faltantes se realizó sobre las vías vehiculares y posteriormente se verificó el resultado directamente en SUMO.


## 5. Validación de la red SUMO

La red fue cargada en NetEdit:

```bash
netedit data/processed/usaquen.net.xml
```

NetEdit reportó:

```text
Import done:
   12989 nodes loaded.
   19 types loaded.
   28070 edges loaded.
```

Por lo tanto, la red final contiene:

* **12.989 nodos**
* **28.070 edges**
* **19 tipos de vía**

La cantidad de edges coincide con la red cargada en NetEdit.

## 6. Velocidades y carriles en SUMO

Se analizaron los atributos finales presentes en los edges de SUMO.

### Velocidades

La red presenta velocidades entre:

```text
5 km/h y 100 km/h
```

Estadísticas:

```text
Media:          28,38 km/h
Mediana:        20,02 km/h
Percentil 25:   10,01 km/h
Percentil 75:   29,99 km/h
Máximo:        100,01 km/h
```

### Carriles

Distribución de carriles por edge:

| Carriles |  Edges |
| -------: | -----: |
|        1 | 25.174 |
|        2 |  2.022 |
|        3 |    754 |
|        4 |     89 |
|        5 |     19 |
|        6 |      9 |
|        7 |      1 |
|       10 |      2 |

Esto confirma que los edges de SUMO cuentan con una configuración de carriles incluso cuando el atributo `lanes` original de OSM estaba ausente.


## 7. Tipos de vía generados en SUMO

La red final contiene 19 tipos:

| Tipo SUMO              | Edges |
| ---------------------- | ----: |
| highway.residential    | 8.727 |
| highway.footway        | 7.367 |
| highway.service        | 5.870 |
| highway.primary        | 1.326 |
| highway.cycleway       | 1.187 |
| highway.secondary      |   985 |
| highway.tertiary       |   968 |
| highway.pedestrian     |   368 |
| highway.path           |   276 |
| highway.steps          |   237 |
| highway.trunk          |   235 |
| highway.primary_link   |   164 |
| highway.track          |   162 |
| highway.trunk_link     |    95 |
| highway.unclassified   |    77 |
| highway.secondary_link |    18 |
| highway.bridleway      |     4 |
| highway.tertiary_link  |     3 |
| highway.service|psv    |     1 |

La conversión conserva tanto infraestructura motorizada como infraestructura destinada a peatones y bicicletas.


## 8. Accesibilidad multimodal

Se revisaron los atributos `allow` y `disallow` generados por SUMO.

### Principales valores de `allow`

```text
pedestrian
pedestrian delivery bicycle
pedestrian bicycle
bicycle
pedestrian motorcycle moped bicycle
bus bicycle
emergency authority bus bicycle
delivery bicycle
```

### Principales valores de `disallow`

```text
tram rail_urban rail rail_electric rail_fast ...
pedestrian ...
pedestrian bicycle ...
all
```

Estos atributos permiten que SUMO determine qué tipos de vehículos o actores pueden utilizar cada edge.

La configuración observada es coherente con una red multimodal, donde una vía puede estar destinada exclusivamente a peatones, bicicletas o permitir diferentes combinaciones de usuarios.


## 9. Aceras y cruces peatonales

Se revisaron los atributos de aceras presentes en OSM.

### `sidewalk`

```text
NaN:         11.936
separate:       314
both:            38
right:           14
no:                8
```

### `sidewalk:right`

```text
NaN:         12.298
separate:       10
no:               2
```

### `sidewalk:left`

```text
NaN:         12.299
no:               9
separate:         2
```

### `sidewalk:both`

```text
NaN:         12.309
yes:               1
```

También se identificaron:

```text
Cruces peatonales registrados en OSM: 2.134
```

La red fue revisada visualmente en NetEdit, verificando la presencia de infraestructura peatonal y la configuración multimodal.


## 10. Validación final

La validación final de la red produjo:

```text
VALIDACIÓN FINAL DE LA RED:

Nodos cargados en NetEdit: 12.989
Edges SUMO: 28.070
Tipos de vía SUMO: 19
Cruces OSM registrados: 2.134
```

La red fue revisada visualmente en NetEdit.

Se verificó que:

* La red se encuentra conectada.
* Las vías vehiculares cuentan con velocidades y carriles en la red SUMO.
* Las vías no motorizadas cuentan con configuraciones apropiadas para sus respectivos modos.
* Existen restricciones de acceso mediante `allow` y `disallow`.
* Se conservan vías peatonales y ciclorrutas.
* Se encuentran presentes atributos relacionados con infraestructura peatonal.
* La red final puede ser utilizada como entrada para la siguiente etapa de configuración de la simulación.


## 11. Archivos generados

Los principales archivos generados durante esta etapa son:

```text
data/processed/usaquen.osm.xml
data/processed/usaquen.net.xml
```

### `usaquen.osm.xml`

Red vial obtenida desde OSM mediante Overpass API utilizando como límite el polígono oficial de Usaquén.

### `usaquen.net.xml`

Red convertida al formato nativo de SUMO mediante `netconvert`, incluyendo la configuración multimodal necesaria para la simulación.


### Resultado
La red vial de Usaquén fue convertida exitosamente desde OSM a SUMO y validada visualmente en NetEdit. La red final contiene **28.070 edges, 12.989 nodos y 19 tipos de vía**, incorporando infraestructura para vehículos, bicicletas y peatones.

El análisis de atributos originales de OSM mostró que una proporción importante de las vías vehiculares no tenía explícitamente `maxspeed` o `lanes`. Sin embargo, `netconvert` generó una red SUMO con valores de velocidad, carriles y restricciones de acceso para los edges resultantes.

La red se encuentra, por tanto, preparada para continuar con la siguiente etapa de configuración de la simulación.

**Nota:** Que haya velocidades y carriles en SUMO no implica que dichos valores estén calibrados para representar el comportamiento real de Bogotá. La calibración de velocidades, capacidades y comportamiento de los actores frente a datos observados corresponde a etapas posteriores de configuración y validación de la simulación.